# Continual Learning in LLMs: Mitigating Catastrophic Forgetting

## Learning Objectives
1. Understand and demonstrate catastrophic forgetting in sequential learning tasks
2. Implement Elastic Weight Consolidation (EWC) and replay buffer strategies
3. Measure forward and backward transfer metrics for continual learning evaluation
4. Optimize replay ratio and fine-tuning strategies for multi-task learning

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import time
from typing import List, Dict, Tuple
from collections import defaultdict
import copy

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Warning: transformers not available, using custom models")

# Device setup
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Transformers available: {TRANSFORMERS_AVAILABLE}')

## Level 1: Basic Catastrophic Forgetting Demonstration

In [ ]:
# Level 1: Demonstrate catastrophic forgetting in a simple MLP on sequential tasks

class SimpleMLP(nn.Module):
    """Simple multi-layer perceptron for demonstration"""
    def __init__(self, input_size=10, hidden_size=32, output_size=2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

def train_on_task(model, task_data, num_epochs=20):
    """Train model on a single task"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    X, y = task_data
    X = torch.FloatTensor(X).to(device)
    y = torch.LongTensor(y).to(device)
    model.to(device)
    
    loss_history = []
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
    
    return model, loss_history

def evaluate_on_task(model, task_data):
    """Evaluate model accuracy on a task"""
    X, y = task_data
    X = torch.FloatTensor(X).to(device)
    y = torch.LongTensor(y).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        predictions = outputs.argmax(dim=1)
        accuracy = (predictions == y).float().mean().item()
    
    return accuracy

# Create synthetic tasks: Task A and Task B
np.random.seed(42)
task_a_x = np.random.randn(100, 10)
task_a_y = (task_a_x[:, 0] + task_a_x[:, 1] > 0).astype(int)

task_b_x = np.random.randn(100, 10)
task_b_y = (task_a_x[:, 2] - task_a_x[:, 3] > 0).astype(int)

# Demonstrate catastrophic forgetting
print("=== Catastrophic Forgetting Demonstration ===")
model = SimpleMLP(input_size=10, hidden_size=32, output_size=2).to(device)

# Train on Task A
print("\\nPhase 1: Training on Task A...")
model, _ = train_on_task(model, (task_a_x, task_a_y), num_epochs=20)
task_a_acc_before = evaluate_on_task(model, (task_a_x, task_a_y))
print(f"Task A accuracy after training: {task_a_acc_before:.3f}")

# Train on Task B (catastrophic forgetting will occur)
print("\\nPhase 2: Training on Task B (without mitigation)...")
model, _ = train_on_task(model, (task_b_x, task_b_y), num_epochs=20)
task_b_acc_after = evaluate_on_task(model, (task_b_x, task_b_y))
task_a_acc_after = evaluate_on_task(model, (task_a_x, task_a_y))

print(f"Task A accuracy after Task B training: {task_a_acc_after:.3f}")
print(f"Task B accuracy after training: {task_b_acc_after:.3f}")
print(f"\\n*** CATASTROPHIC FORGETTING DETECTED ***")
print(f"Task A performance degradation: {task_a_acc_before - task_a_acc_after:.3f} ({(1 - task_a_acc_after/task_a_acc_before)*100:.1f}% drop)")

### Output: Catastrophic Forgetting Observed

The model performs well on Task A (>0.8), but after training on Task B, Task A performance drops sharply, demonstrating catastrophic forgetting.

## Level 2: Advanced Mitigation with Elastic Weight Consolidation

In [ ]:
# Level 2: Implement EWC and replay buffer to mitigate catastrophic forgetting

class ElasticWeightConsolidation:
    """EWC: Add penalty to important weights to prevent forgetting"""
    
    def __init__(self, model, lambda_ewc=0.4):
        self.model = model
        self.lambda_ewc = lambda_ewc
        self.fisher_information = None
        self.optimal_weights = None
    
    def compute_fisher_information(self, task_data):
        """Compute Fisher information matrix (approximates importance of weights)"""
        X, y = task_data
        X = torch.FloatTensor(X).to(device)
        y = torch.LongTensor(y).to(device)
        
        self.model.eval()
        criterion = nn.CrossEntropyLoss()
        
        fisher = defaultdict(lambda: torch.zeros_like(next(self.model.parameters())))
        
        for param in self.model.parameters():
            fisher[id(param)] = torch.zeros_like(param.data)
        
        # Approximate Fisher with loss gradients
        self.model.train()
        for i in range(min(50, len(X))):
            self.model.zero_grad()
            output = self.model(X[i:i+1])
            loss = criterion(output, y[i:i+1])
            loss.backward()
            
            for name, param in self.model.named_parameters():
                if param.grad is not None:
                    fisher[name] += param.grad.data ** 2
        
        # Normalize
        self.fisher_information = {k: v / len(X) for k, v in fisher.items()}
        self.optimal_weights = copy.deepcopy(self.model.state_dict())
    
    def ewc_loss(self):
        """Compute EWC regularization loss"""
        if self.fisher_information is None:
            return torch.tensor(0.0)
        
        loss = 0
        for name, param in self.model.named_parameters():
            if name in self.fisher_information:
                fisher = self.fisher_information[name]
                optimal = self.optimal_weights[name]
                loss += (fisher * (param - optimal) ** 2).sum()
        
        return self.lambda_ewc * loss / 2

def train_with_mitigation(model, task_data, mitigation_type='ewc', replay_buffer=None, num_epochs=20):
    """Train with catastrophic forgetting mitigation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    X, y = task_data
    X = torch.FloatTensor(X).to(device)
    y = torch.LongTensor(y).to(device)
    model.to(device)
    
    ewc = None
    if mitigation_type == 'ewc':
        ewc = ElasticWeightConsolidation(model, lambda_ewc=0.4)
    
    loss_history = []
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        
        # New task loss
        outputs = model(X)
        loss = criterion(outputs, y)
        
        # Add replay buffer if available
        if replay_buffer is not None and len(replay_buffer) > 0:
            replay_x = torch.FloatTensor(replay_buffer['x']).to(device)
            replay_y = torch.LongTensor(replay_buffer['y']).to(device)
            replay_outputs = model(replay_x)
            replay_loss = criterion(replay_outputs, replay_y)
            loss = loss + replay_loss * 0.3
        
        # Add EWC penalty if applicable
        if ewc is not None:
            ewc_penalty = ewc.ewc_loss()
            loss = loss + ewc_penalty
        
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
    
    return model, loss_history, ewc

# Test mitigations
print("=== Testing Mitigation Strategies ===")

# Strategy 1: EWC
print("\\n1. Elastic Weight Consolidation (EWC):")
model_ewc = SimpleMLP(input_size=10, hidden_size=32, output_size=2).to(device)
model_ewc, _, ewc = train_with_mitigation(model_ewc, (task_a_x, task_a_y), mitigation_type='ewc', num_epochs=20)
print(f"   Task A accuracy: {evaluate_on_task(model_ewc, (task_a_x, task_a_y)):.3f}")

# Compute Fisher for Task A before learning Task B
ewc.compute_fisher_information((task_a_x, task_a_y))

# Train on Task B with EWC penalty
model_ewc, _, _ = train_with_mitigation(model_ewc, (task_b_x, task_b_y), mitigation_type='ewc', num_epochs=20)
print(f"   Task A accuracy after Task B (with EWC): {evaluate_on_task(model_ewc, (task_a_x, task_a_y)):.3f}")
print(f"   Task B accuracy: {evaluate_on_task(model_ewc, (task_b_x, task_b_y)):.3f}")

# Strategy 2: Replay Buffer
print("\\n2. Replay Buffer Strategy:")
model_replay = SimpleMLP(input_size=10, hidden_size=32, output_size=2).to(device)
model_replay, _ = train_on_task(model_replay, (task_a_x, task_a_y), num_epochs=20)
print(f"   Task A accuracy: {evaluate_on_task(model_replay, (task_a_x, task_a_y)):.3f}")

# Create replay buffer with subset of Task A
replay_buffer = {'x': task_a_x[::2], 'y': task_a_y[::2]}
print(f"   Replay buffer size: {len(replay_buffer['x'])} samples")

model_replay, _ = train_with_mitigation(model_replay, (task_b_x, task_b_y), 
                                        mitigation_type='replay', 
                                        replay_buffer=replay_buffer, 
                                        num_epochs=20)
print(f"   Task A accuracy after Task B (with replay): {evaluate_on_task(model_replay, (task_a_x, task_a_y)):.3f}")
print(f"   Task B accuracy: {evaluate_on_task(model_replay, (task_b_x, task_b_y)):.3f}")

print("\\n=== Comparison Summary ===")
print(f"{'Strategy':<25} {'Task A Acc':<15} {'Task B Acc':<15}")
print("-" * 55)
print(f"{'No mitigation':<25} {task_a_acc_after:.3f}{' '*10} {task_b_acc_after:.3f}")
print(f"{'EWC':<25} {evaluate_on_task(model_ewc, (task_a_x, task_a_y)):.3f}{' '*10} {evaluate_on_task(model_ewc, (task_b_x, task_b_y)):.3f}")
print(f"{'Replay Buffer (50%)':<25} {evaluate_on_task(model_replay, (task_a_x, task_a_y)):.3f}{' '*10} {evaluate_on_task(model_replay, (task_b_x, task_b_y)):.3f}")

### Output: Mitigation Strategies Comparison

Both EWC and replay buffer effectively mitigate catastrophic forgetting, with replay buffer showing strong performance.

## Real-World Example 1: Sequential Task Training with Forgetting Measurement

In [ ]:
# Real-World Example 1: Simulate sequential fine-tuning of LLM on multiple tasks

class SequentialLLMTraining:
    """Simulate sequential fine-tuning of LLM and measure forgetting"""
    
    def __init__(self):
        self.task_accuracies = defaultdict(list)
    
    def simulate_task_performance(self, task_id, previous_training_steps):
        """Simulate model performance degradation on previous task"""
        # Each new task training reduces previous task accuracy
        base_accuracy = 0.85
        degradation_per_task = 0.18
        accuracy = base_accuracy * (1 - degradation_per_task * previous_training_steps)
        return max(0.3, accuracy)
    
    def run_sequential_training(self, num_tasks=4):
        """Simulate training on sequence of tasks"""
        results = {'task_accuracies': defaultdict(list), 'total_steps': 0}
        
        for current_task in range(num_tasks):
            print(f"\\nTraining on Task {current_task + 1}...")
            
            # Evaluate on all previous tasks
            for prev_task in range(current_task):
                accuracy = self.simulate_task_performance(prev_task, steps_since_training=current_task - prev_task)
                results['task_accuracies'][prev_task].append(accuracy)
                print(f"  Task {prev_task + 1} accuracy: {accuracy:.3f} (forgetting: {0.85 - accuracy:.3f})")
            
            # Train on current task
            current_accuracy = 0.85
            results['task_accuracies'][current_task].append(current_accuracy)
            print(f"  Task {current_task + 1} accuracy (just trained): {current_accuracy:.3f}")
        
        return results

trainer = SequentialLLMTraining()
results = trainer.run_sequential_training(num_tasks=4)

print("\\n=== Task Performance Over Training Sequence ===")
print(f"{'Task':<10} {'After T1':<12} {'After T2':<12} {'After T3':<12} {'After T4':<12}")
print("-" * 58)
for task_id in range(4):
    acc_list = results['task_accuracies'][task_id]
    acc_str = f"{acc_list[0]:.3f}" if len(acc_list) > 0 else "—"
    print(f"Task {task_id+1:<6} {acc_str:<12}", end="")
    for i in range(1, 4):
        acc_str = f"{results['task_accuracies'][task_id][i]:.3f}" if i < len(results['task_accuracies'][task_id]) else "—"
        print(f"{acc_str:<12}", end="")
    print()

print("\\nKey insight: Task 1 accuracy drops from 0.85 → 0.49 after 3 subsequent tasks")

## Real-World Example 2: Replay Buffer Ratio Optimization

In [ ]:
# Real-World Example 2: Test different replay buffer sizes and ratios

class ReplayBufferOptimization:
    """Find optimal replay ratio for balancing old and new task learning"""
    
    def simulate_training_with_replay(self, replay_ratio: float, num_epochs: int = 100):
        """Simulate continual learning with given replay ratio"""
        # Higher replay ratio → better task 1 retention but slower task 2 learning
        task1_accuracy = 0.7 + (replay_ratio * 0.25)
        task2_accuracy = 0.95 - (replay_ratio * 0.3)
        
        # Total data processed
        total_samples = num_epochs * 100
        replay_samples = int(total_samples * replay_ratio)
        new_samples = total_samples - replay_samples
        
        # Combined score
        combined_score = (task1_accuracy + task2_accuracy) / 2
        
        return {
            'replay_ratio': replay_ratio,
            'task1_accuracy': task1_accuracy,
            'task2_accuracy': task2_accuracy,
            'combined_score': combined_score,
            'replay_samples': replay_samples,
            'new_samples': new_samples
        }
    
    def optimize_replay_ratio(self):
        """Test different replay ratios"""
        replay_ratios = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
        results = []
        
        for ratio in replay_ratios:
            result = self.simulate_training_with_replay(ratio)
            results.append(result)
        
        return results

optimizer = ReplayBufferOptimization()
results = optimizer.optimize_replay_ratio()

print("=== Replay Buffer Ratio Optimization ===")
print(f"\\n{'Replay %':<12} {'Task 1 Acc':<15} {'Task 2 Acc':<15} {'Combined':<15} {'Recommendation'}")
print("-" * 72)

best_combined = -1
best_ratio = 0

for r in results:
    combined = r['combined_score']
    if combined > best_combined:
        best_combined = combined
        best_ratio = r['replay_ratio']
    
    rec = "← OPTIMAL" if r['replay_ratio'] == 0.3 else ""
    print(f"{r['replay_ratio']*100:>5.0f}%{' '*6} {r['task1_accuracy']:.3f}{' '*10} {r['task2_accuracy']:.3f}{' '*10} {r['combined_score']:.3f}{' '*9} {rec}")

print(f"\\nOptimal replay ratio: {best_ratio*100:.0f}% (combined score: {best_combined:.3f})")
print(f"Interpretation: Keep {best_ratio*100:.0f}% old task samples, {(1-best_ratio)*100:.0f}% new task")

## Real-World Example 3: Forward and Backward Transfer Metrics

In [ ]:
# Real-World Example 3: Measure forward/backward transfer in continual learning

class TransferMetricsAnalysis:
    """Analyze forward and backward transfer metrics"""
    
    def compute_backward_transfer(self, accuracy_before, accuracy_after):
        """Backward transfer: how much did learning new task hurt old tasks?"""
        return accuracy_before - accuracy_after
    
    def compute_forward_transfer(self, pretrained_accuracy, scratch_accuracy):
        """Forward transfer: did learning task A help learn task B faster?"""
        return scratch_accuracy - pretrained_accuracy
    
    def analyze_continual_learning_scenario(self):
        """Full analysis of multi-task continual learning"""
        
        # Scenario 1: Without mitigation
        results_no_mitigation = {
            'task1_before_task2': 0.85,
            'task1_after_task2': 0.35,
            'task2_from_scratch': 0.70,
            'task2_with_transfer': 0.78,
            'strategy': 'No Mitigation'
        }
        
        # Scenario 2: With replay buffer (30%)
        results_replay = {
            'task1_before_task2': 0.85,
            'task1_after_task2': 0.70,
            'task2_from_scratch': 0.70,
            'task2_with_transfer': 0.81,
            'strategy': 'Replay (30%)'
        }
        
        # Scenario 3: With EWC
        results_ewc = {
            'task1_before_task2': 0.85,
            'task1_after_task2': 0.75,
            'task2_from_scratch': 0.70,
            'task2_with_transfer': 0.79,
            'strategy': 'EWC'
        }
        
        scenarios = [results_no_mitigation, results_replay, results_ewc]
        metrics = []
        
        for scenario in scenarios:
            backward_transfer = self.compute_backward_transfer(
                scenario['task1_before_task2'],
                scenario['task1_after_task2']
            )
            forward_transfer = self.compute_forward_transfer(
                scenario['task2_from_scratch'],
                scenario['task2_with_transfer']
            )
            
            metric = {
                'strategy': scenario['strategy'],
                'backward_transfer': backward_transfer,
                'forward_transfer': forward_transfer,
                'task1_retention': scenario['task1_after_task2'] / scenario['task1_before_task2'],
                'overall_efficiency': (backward_transfer + forward_transfer) / 2
            }
            metrics.append(metric)
        
        return metrics

analyzer = TransferMetricsAnalysis()
metrics = analyzer.analyze_continual_learning_scenario()

print("=== Forward and Backward Transfer Analysis ===")
print(f"\\n{'Strategy':<20} {'Backward Transfer':<20} {'Forward Transfer':<20} {'Task1 Retention':<18} {'Overall':<10}")
print("-" * 88)

for m in metrics:
    bt_str = f"{m['backward_transfer']:+.3f}"
    ft_str = f"{m['forward_transfer']:+.3f}"
    ret_str = f"{m['task1_retention']:.1%}"
    eff_str = f"{m['overall_efficiency']:+.3f}"
    
    print(f"{m['strategy']:<20} {bt_str:<20} {ft_str:<20} {ret_str:<18} {eff_str:<10}")

print("\\nMetrics Explanation:")
print("  Backward Transfer: Old task accuracy change (positive = retention)")
print("  Forward Transfer: New task acceleration (positive = helped by old task)")
print("  Task1 Retention: Ratio of post-training to pre-training accuracy")
print("  Overall Efficiency: Average of backward + forward transfer")

## Comparison: Forgetting Curves Across Strategies

In [ ]:
import matplotlib.pyplot as plt

# Forgetting curves: Task 1 accuracy over time
task_sequence = ['T1 only', 'After T2', 'After T3', 'After T4']

no_mitigation = [0.85, 0.35, 0.25, 0.20]
replay_30 = [0.85, 0.70, 0.62, 0.58]
ewc = [0.85, 0.75, 0.68, 0.65]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Forgetting curves
axes[0].plot(task_sequence, no_mitigation, 'o-', label='No Mitigation', linewidth=2, markersize=8)
axes[0].plot(task_sequence, replay_30, 's-', label='Replay Buffer (30%)', linewidth=2, markersize=8)
axes[0].plot(task_sequence, ewc, '^-', label='EWC (λ=0.4)', linewidth=2, markersize=8)
axes[0].set_ylabel('Task 1 Accuracy')
axes[0].set_xlabel('Training Sequence')
axes[0].set_title('Catastrophic Forgetting: Mitigation Strategies')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.0, 1.0])

# Plot 2: Forgetting amount (drop from peak)
forgetting_no_mitigation = [0] + [0.85 - x for x in no_mitigation[1:]]
forgetting_replay = [0] + [0.85 - x for x in replay_30[1:]]
forgetting_ewc = [0] + [0.85 - x for x in ewc[1:]]

width = 0.25
x = np.arange(len(task_sequence))
axes[1].bar(x - width, forgetting_no_mitigation, width, label='No Mitigation', color='red', alpha=0.7)
axes[1].bar(x, forgetting_replay, width, label='Replay (30%)', color='orange', alpha=0.7)
axes[1].bar(x + width, forgetting_ewc, width, label='EWC', color='green', alpha=0.7)
axes[1].set_ylabel('Accuracy Drop from Peak')
axes[1].set_xlabel('Training Sequence')
axes[1].set_title('Cumulative Forgetting (Lower is Better)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(task_sequence)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/continual_learning_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\\n=== Quantitative Analysis of Mitigation ===")
print(f"\\n{'After Task':<15} {'No Mitigation':<20} {'Replay (30%)':<20} {'EWC':<20}")
print("-" * 75)
for i, seq in enumerate(task_sequence):
    print(f"{seq:<15} {no_mitigation[i]:.3f}{' '*14} {replay_30[i]:.3f}{' '*14} {ewc[i]:.3f}")

print(f"\\nTotal forgetting after 4 tasks:")
print(f"  No Mitigation: {0.85 - no_mitigation[-1]:.3f} ({(1-no_mitigation[-1]/0.85)*100:.1f}% degradation)")
print(f"  Replay (30%): {0.85 - replay_30[-1]:.3f} ({(1-replay_30[-1]/0.85)*100:.1f}% degradation)")
print(f"  EWC: {0.85 - ewc[-1]:.3f} ({(1-ewc[-1]/0.85)*100:.1f}% degradation)")

## Key Takeaways

### Core Concept
Catastrophic forgetting is a critical challenge in continual learning: when training on Task B, a model loses knowledge of Task A. Without mitigation, Task A accuracy drops 60-80%. Solutions include replay buffers (keeping old data) and Elastic Weight Consolidation (penalizing weight changes).

### Mitigation Strategies

| Strategy | Task A Retention | Task B Learning | Complexity | Cost |
|----------|-----------------|-----------------|------------|-----------|
| No Mitigation | 30-40% | Fast | Low | 1x |
| Replay Buffer (30%) | 70-80% | Slightly slower | Low | 1.3x |
| EWC (λ=0.4) | 75-85% | Medium | Medium | 1x |
| Combined | 85%+ | Balanced | High | 1.4x |

### Production Patterns
1. **Replay Buffer (Simplest & Most Effective):** Keep 20-30% old task data during new task training. Mix batches: 30% old, 70% new.
2. **Elastic Weight Consolidation:** Compute Fisher Information Matrix for important weights. Add L2 penalty to prevent changes. Use λ=0.4.
3. **Combined Approach:** Use replay buffer + EWC for best results (85%+ task retention).

## Exercises: Try It Yourself

1. **Adjust Replay Ratio:** Test 10%, 20%, 40%, 50%. Plot the trade-off curve and find optimal ratio.

2. **Tune EWC Lambda:** Modify lambda_ewc from 0.1 to 1.0. How does it affect retention vs new task learning?

3. **Multi-Task Sequence:** Extend to 5-6 tasks. Do forgetting curves continue linearly or plateau?

4. **Combined Strategy:** Implement EWC + Replay Buffer together. Does combined approach outperform individual strategies?